# 04 Synthesis

Display layer for double compression. This notebook reads only tidy tables from `02` and `03`; it does not import `diversity_facets` and does not recompute metrics.

In [1]:
CONFIG = dict(
    conditions=["baseline", "one_at_a_time", "persona"],
    text_versions=["rephrased", "original"],
    comparisons=["human_vs_claude", "human_vs_gemini", "human_vs_gpt", "human_vs_pooled_ai"],
)

# Facet -> (metric, param per task). Coverage's param differs by task; evenness is
# carried on the aligned standardized axes from the fingerprint CSVs (redesign spec 4.3).
FACET_SPECS = {
    "spread": ("mean_pairwise", {"proposals": "", "reviews": ""}),
    "evenness": ("ripley_excess", {"proposals": "r=pooled_q01_q50", "reviews": "r=pooled_q01_q50"}),
    "richness": ("vendi", {"proposals": "q=1", "reviews": "q=1"}),
    "dimensionality": ("participation_ratio", {"proposals": "", "reviews": ""}),
    "coverage": ("coverage_geometric", {"proposals": "k=3", "reviews": "k=panel_adaptive"}),
}
# fig2 headline = richness (Vendi q=1); supplements per spec 15.3 (M0, M2, M3 + the
# evenness slope as M4's ratio-able carrier).
FIG1_SUPPLEMENTS = {
    "spread": ("mean_pairwise", {"proposals": "", "reviews": ""}),
    "coverage": ("coverage_geometric", {"proposals": "k=3", "reviews": "k=panel_adaptive"}),
    "dimensionality": ("participation_ratio", {"proposals": "", "reviews": ""}),
    "evenness": ("vendi_slope", {"proposals": "q=0..2", "reviews": "q=0..2"}),
}
CONDITION_MARKERS = {"baseline": "o", "one_at_a_time": "s", "persona": "^"}
WRITE_FIGURES = True

## Load

Read completed `facet_diversity_tests.csv` files from proposals and reviews. M5 displacement is excluded from ratio figures by design.

In [2]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import plotting as pl  # display helpers only; imports no metric code

assert "diversity_facets" not in sys.modules, "04_synthesis must not import or recompute diversity metrics"

def _read_tests(task, condition, text_version):
    path = PROJECT_ROOT / "results" / "tables" / condition / task / text_version / "facet_diversity_tests.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing tidy table from 02/03: {path}")
    df = pd.read_csv(path)
    df["task"] = task
    return df

def _read_fingerprint(task, condition, text_version):
    path = PROJECT_ROOT / "results" / "tables" / condition / task / text_version / "facet_fingerprint.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing fingerprint CSV from 02/03 (redesign spec 3.3): {path}")
    df = pd.read_csv(path)
    df["task"] = task
    df["param"] = df["param"].fillna("")
    df["stars"] = df["stars"].fillna("")
    return df

def _read_interleaving(task, condition, text_version):
    path = PROJECT_ROOT / "results" / "tables" / condition / task / text_version / "facet_interleaving.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing interleaving CSV from 02/03: {path}")
    df = pd.read_csv(path)
    df["task"] = task
    return df

frames, fp_frames, inter_frames = [], [], []
for task in ["proposals", "reviews"]:
    for condition in CONFIG["conditions"]:
        for text_version in CONFIG["text_versions"]:
            frames.append(_read_tests(task, condition, text_version))
            fp_frames.append(_read_fingerprint(task, condition, text_version))
            inter_frames.append(_read_interleaving(task, condition, text_version))
T = pd.concat(frames, ignore_index=True)
T = T[T["comparison"].isin(CONFIG["comparisons"])].copy()
FP = pd.concat(fp_frames, ignore_index=True)
INTER = pd.concat(inter_frames, ignore_index=True)
assert "diversity_facets" not in sys.modules, "loading tidy tables must not pull in metric code"
print(f"tests rows: {len(T)}; fingerprint rows: {len(FP)}")
T.head()

tests rows: 1320; fingerprint rows: 300


,condition,task,text_version,field,comparison,facet,metric,is_primary,param,human_value,...,human_ci_hi,inference,stat,p_raw,p_fdr,n_human,n_ai,n_perm_or_sub,parity_ref,notes
0,baseline,proposals,rephrased,whole,human_vs_claude,spread,mean_pairwise,True,NaN,0.415570,...,0.424403,permutation,0.160459,0.079292,NaN,23.0,23.0,10000.0,1.0,"spread facet; ci = 95% jackknife (AI group), h..."
1,baseline,proposals,rephrased,whole,human_vs_claude,spread,centroid_loo,False,NaN,0.640121,...,0.651821,permutation,0.217616,0.046395,0.139186,23.0,23.0,10000.0,1.0,"spread facet; ci = 95% jackknife (AI group), h..."
2,baseline,proposals,rephrased,whole,human_vs_claude,spread,mst_dispersion,False,NaN,0.110074,...,0.114508,permutation,0.030784,0.066293,0.172498,23.0,23.0,10000.0,1.0,"spread facet; ci = 95% jackknife (AI group), h..."
3,baseline,proposals,rephrased,whole,human_vs_claude,spread,sparseness,False,NaN,0.322715,...,0.336489,permutation,0.163548,0.062494,0.165999,23.0,23.0,10000.0,1.0,"spread facet; ci = 95% jackknife (AI group), h..."
4,baseline,proposals,rephrased,whole,human_vs_claude,spread,nn_isolation,False,NaN,0.081982,...,0.085081,permutation,0.028900,0.073293,0.186896,23.0,23.0,10000.0,1.0,"spread facet; ci = 95% jackknife (AI group), h..."


## Ratios

Compute the single normalized quantity used by synthesis figures: AI ÷ Human diversity retained. Coverage parity uses the exported split-half reference instead of assuming 1.0.

In [3]:
# The single normalized quantity (spec 15.2). M5/displacement is excluded (no ratio);
# the lexical control and the density companion are not facet metrics.
ratio_df = T[~T["facet"].isin(["displacement", "lexical_control"])].copy()
ratio_df = ratio_df[ratio_df["metric"].ne("coverage_density")]
ratio_df["param"] = ratio_df["param"].fillna("")
ratio_df = ratio_df[np.isfinite(ratio_df["human_value"]) & np.isfinite(ratio_df["ai_value"])]
ratio_df["ratio"] = ratio_df["ai_value"] / ratio_df["human_value"]
ratio_df["log2ratio"] = np.log2(ratio_df["ratio"].where(ratio_df["ratio"] > 0))
ratio_df["parity_ref"] = ratio_df.get("parity_ref", 1.0).fillna(1.0)
coverage_mask = ratio_df["metric"].eq("coverage_geometric")
assert ratio_df.loc[coverage_mask, "parity_ref"].notna().all(), "coverage parity_ref must be exported by 02/03"
# Coverage human_value IS the split-half / LOO parity reference, so parity in ratio
# space is 1.0 for every metric here (spec 15.2 note).
ratio_df.head()

,condition,task,text_version,field,comparison,facet,metric,is_primary,param,human_value,...,stat,p_raw,p_fdr,n_human,n_ai,n_perm_or_sub,parity_ref,notes,ratio,log2ratio
0,baseline,proposals,rephrased,whole,human_vs_claude,spread,mean_pairwise,True,,0.415570,...,0.160459,0.079292,NaN,23.0,23.0,10000.0,1.0,"spread facet; ci = 95% jackknife (AI group), h...",0.613881,-0.703969
1,baseline,proposals,rephrased,whole,human_vs_claude,spread,centroid_loo,False,,0.640121,...,0.217616,0.046395,0.139186,23.0,23.0,10000.0,1.0,"spread facet; ci = 95% jackknife (AI group), h...",0.660040,-0.599375
2,baseline,proposals,rephrased,whole,human_vs_claude,spread,mst_dispersion,False,,0.110074,...,0.030784,0.066293,0.172498,23.0,23.0,10000.0,1.0,"spread facet; ci = 95% jackknife (AI group), h...",0.720338,-0.473255
3,baseline,proposals,rephrased,whole,human_vs_claude,spread,sparseness,False,,0.322715,...,0.163548,0.062494,0.165999,23.0,23.0,10000.0,1.0,"spread facet; ci = 95% jackknife (AI group), h...",0.493213,-1.019718
4,baseline,proposals,rephrased,whole,human_vs_claude,spread,nn_isolation,False,,0.081982,...,0.028900,0.073293,0.186896,23.0,23.0,10000.0,1.0,"spread facet; ci = 95% jackknife (AI group), h...",0.647491,-0.627068


## Figures

Six figures per `text_version` (redesign spec §4.1): **fig1** = the direction-aligned diversity fingerprint hero (schematic reading key + generation z-fingerprints + filtering δ-fingerprints, humans/parity at 0, right = more diverse everywhere); **fig2** slopegraph; **fig3** compression map; **fig4** robustness grid with the evenness row re-expressed on the aligned standardized axes; **fig5** paired UMAPs; **fig6** condition gradient. Old fig-numbered files are removed so the directory carries one consistent set. PNAS presets and redundant group marker shapes throughout.

In [4]:
PARITY = "#404040"
OLD_FIG_STEMS = ["fig1_double_compression_slopegraph", "fig2_compression_map", "fig3_robustness_grid",
                 "fig4_paired_umaps_baseline", "fig4_paired_umaps_one_at_a_time", "fig4_paired_umaps_persona",
                 "fig5_condition_gradient"]


def _model_label(comparison):
    return pl.COMPARISON_TO_GROUP.get(comparison, comparison)


def _whole_only(df):
    return df[df["field"].eq("whole")].copy() if "field" in df.columns else df.copy()


def _metric_slice(df, metric, params_by_task):
    """Rows for one metric with the task-appropriate param (coverage differs by task)."""
    whole = _whole_only(df)
    parts = []
    for task, param in params_by_task.items():
        parts.append(whole[whole["task"].eq(task) & whole["metric"].eq(metric) & whole["param"].eq(param)])
    return pd.concat(parts, ignore_index=True) if parts else whole.iloc[0:0]


def _axis_mode(values):
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    return "log2" if (vals > 1.0).any() else "linear"


def _transform(vals, mode):
    return np.log2(vals) if mode == "log2" else vals


def _caption(fig, text):
    # Wrap long captions: an unwrapped single line widens the tight bounding box and
    # pads the exported figure with whitespace.
    import textwrap
    fig.text(0.01, -0.02, "\n".join(textwrap.wrap(text, 170)), ha="left", va="top",
             fontsize=7.5, color="#333333")


def fig1_fingerprint(fp, out, text_version):
    """Hero figure (redesign spec 4.2): generation/filtering fingerprints."""
    fig = plt.figure(figsize=(13, 10.5))
    gs = fig.add_gridspec(2, 3, height_ratios=[1.25, 1.05], hspace=0.52, wspace=0.08, top=0.85, bottom=0.06)
    for row, (task, mode, row_title) in enumerate(
            [("proposals", "z", "GENERATION \u2014 proposal diversity vs pooled-cloud chance (z, SD units)"),
             ("reviews", "delta", "FILTERING \u2014 paired review-panel effects (sign-aligned Cliff's \u03b4)")]):
        for ci, condition in enumerate(CONFIG["conditions"]):
            ax = fig.add_subplot(gs[row, ci])
            sub = fp[fp["task"].eq(task) & fp["condition"].eq(condition) & fp["text_version"].eq(text_version)]
            if sub.empty:
                ax.text(0.5, 0.5, "no rows", ha="center", va="center")
                ax.set_axis_off()
                continue
            pl.draw_fingerprint(ax, sub, mode=mode, show_row_labels=(ci == 0), marker_size=55)
            ax.set_title(condition, fontsize=10)
            if ci == 0:
                ax.text(-0.52, 1.10, row_title, transform=ax.transAxes,
                        fontsize=10, fontweight="bold", ha="left")
            if ci > 0:
                ax.set_ylabel("")
    handles = [plt.Line2D([], [], marker=pl.GROUP_MARKERS[g], linestyle="", markersize=8,
                          color=pl.PALETTE[g], markeredgecolor="black", markeredgewidth=0.5,
                          markerfacecolor="none" if g == "All AI" else pl.PALETTE[g],
                          label=f"{g} (n=23 of 69)" if g == "All AI" else f"{g} (n=23)")
               for g in ["Human", "Claude", "Gemini", "GPT", "All AI"]]
    fig.legend(handles=handles, fontsize=9, ncols=5, loc="upper center", bbox_to_anchor=(0.5, 0.93))
    fig.suptitle(f"Diversity fingerprint \u00b7 every facet, one direction \u00b7 {text_version}", y=0.985, fontsize=13)
    pl.add_direction_badge(fig, "\u2192 more diverse (all panels)")
    _caption(fig, "Right of 0 = more diverse, on every row of every panel (Direction Rule). Generation: z vs M=999 same-n "
                  "pooled-cloud draws (coverage: human split-half null); whiskers = 95% jackknife/subsample CI in z "
                  "units. Filtering: Cliff's \u03b4 (AI \u2212 Human) across 23 matched proposal panels; whiskers = bootstrap 95% CI. "
                  "Evenness enters negated on both (clumping metric). Stars: p_raw for pre-registered primaries, "
                  "p_fdr otherwise \u2014 carried from 02/03, nothing recomputed here. M5 displacement is a directional "
                  "check, not a diversity facet, and is deliberately absent.")
    pl.save_fig(fig, out / "fig1_diversity_fingerprint")


def fig2_slopegraph(df, out, *, metric, params_by_task, label, fname, invert_ratio=False):
    metric_df = _metric_slice(df, metric, params_by_task).copy()
    if invert_ratio:
        # Clumping metric (higher = less even): plot Human/AI so that, like every
        # other panel, BELOW parity = AI less diverse (Direction Rule).
        metric_df["ratio"] = 1.0 / metric_df["ratio"]
    pivot = metric_df.pivot_table(index=["condition", "comparison"], columns="task", values="ratio", aggfunc="mean").reset_index()
    pivot = pivot.dropna(subset=["proposals", "reviews"]) if {"proposals", "reviews"}.issubset(pivot.columns) else pivot.iloc[0:0]
    mode = _axis_mode(np.concatenate([pivot["proposals"].to_numpy(), pivot["reviews"].to_numpy()])) if not pivot.empty else "linear"
    parity_y = 0.0 if mode == "log2" else 1.0
    fig, axes = plt.subplots(1, len(CONFIG["conditions"]), figsize=(14, 4.4), sharey=True)
    for ax, condition in zip(axes, CONFIG["conditions"]):
        sub = pivot[pivot.condition == condition]
        for _, row in sub.iterrows():
            g = _model_label(row["comparison"])
            ys = _transform(np.array([row["proposals"], row["reviews"]]), mode)
            ax.plot([0, 1], ys, marker=pl.GROUP_MARKERS.get(g, "o"), color=pl.PALETTE[g], linewidth=2,
                    linestyle="--" if g == "All AI" else "-", label=g)
        ax.axhline(parity_y, color=PARITY, linestyle="--", linewidth=1)
        ax.annotate("human parity", (0.02, parity_y), textcoords="offset points", xytext=(0, 3),
                    fontsize=7.5, color=PARITY)
        ax.set_xticks([0, 1], ["generation\n(proposals)", "filtering\n(reviews)"])
        ax.set_xlim(-0.25, 1.25)
        ax.set_title(condition)
        ax.grid(axis="y", alpha=0.25)
    ylab = "AI / Human diversity retained" if mode == "linear" else "log₂(AI / Human diversity retained)"
    axes[0].set_ylabel(ylab)
    handles, labels = axes[0].get_legend_handles_labels()
    seen = dict(zip(labels, handles))
    axes[-1].legend(seen.values(), seen.keys(), fontsize=8, loc="best")
    fig.suptitle(f"{label} · double-compression slopegraph", y=1.02)
    pl.add_direction_badge(fig, "↑ more diversity retained")
    _caption(fig, (f"Ratio inverted for this clumping metric (Human/AI of the Vendi-profile drop), so below parity = "
                f"AI less even = less diverse, matching every other panel. " if invert_ratio else "")
              + f"Below parity = narrowing on that end; downward slope = compression compounds; panels rising toward "
                  f"parity left-to-right = persona rescue. Axis mode: {mode} (log2 used iff any ratio > 1, spec 1A.6). "
                  "Per-model n=23 vs Human n=23; pooled AI n=23 subsampled from 69. p-values live in the 02/03 tables.")
    pl.save_fig(fig, out / fname)


FIG2B_SPECS = [
    # (facet, metric, params_by_task, label, negate, natural-unit label, is_headline)
    ("richness", "vendi", {"proposals": "q=1", "reviews": "q=1"}, "Richness — Vendi VS₁", False, "effective distinct items (VS₁)", True),
    ("spread", "mean_pairwise", {"proposals": "", "reviews": ""}, "Spread — mean pairwise", False, "mean pairwise cosine distance", False),
    ("coverage", "coverage_geometric", {"proposals": "k=3", "reviews": "k=panel_adaptive"}, "Coverage — geometric", False, "coverage of the human span", False),
    ("dimensionality", "participation_ratio", {"proposals": "", "reviews": ""}, "Dimensionality — participation ratio", False, "participation ratio", False),
    ("evenness", "ripley_excess", {"proposals": "r=pooled_q01_q50", "reviews": "r=pooled_q01_q50"}, "Evenness — −Ripley excess", True, "evenness vs same-size null (−excess)", False),
]


def fig2b_human_ai(tests, out, *, facet, metric, params, label, negate, unit, fname):
    """Human → AI slopegraph: left anchor = the shared human value, right anchors = each
    model + pooled AI, in the metric's NATURAL units; a downward slope = narrowing at
    that stage. Complements fig2 (which asks whether compression compounds across stages)."""
    fig, axes = plt.subplots(2, len(CONFIG["conditions"]), figsize=(13, 7.8), sharey="row")
    stage_rows = [("proposals", "GENERATION — proposal sets"),
                  ("reviews", "FILTERING — review panels (paired means)")]
    for ti, (task, row_lab) in enumerate(stage_rows):
        for ci, cond in enumerate(CONFIG["conditions"]):
            ax = axes[ti, ci]
            sub = _whole_only(tests)
            sub = sub[sub["task"].eq(task) & sub["facet"].eq(facet) & sub["metric"].eq(metric)
                      & sub["param"].fillna("").eq(params[task]) & sub["condition"].eq(cond)]
            f = -1.0 if negate else 1.0
            human_drawn = False
            for comp in CONFIG["comparisons"]:
                r = sub[sub["comparison"].eq(comp)]
                if r.empty:
                    continue
                r = r.iloc[0]
                if not (np.isfinite(r["human_value"]) and np.isfinite(r["ai_value"])):
                    continue
                g = _model_label(comp)
                hv, av = f * r["human_value"], f * r["ai_value"]
                ax.plot([0, 1], [hv, av], color=pl.PALETTE[g], linewidth=2,
                        linestyle="--" if g == "All AI" else "-", zorder=3)
                ax.plot([1], [av], marker=pl.GROUP_MARKERS.get(g, "o"), color=pl.PALETTE[g],
                        markersize=8, markeredgecolor="black", markeredgewidth=0.5,
                        markerfacecolor="none" if g == "All AI" else pl.PALETTE[g], zorder=4)
                s = _stars_from_row(r)
                if s:
                    ax.annotate(s, (1, av), textcoords="offset points", xytext=(10, -3), fontsize=8)
                if not human_drawn:
                    ax.plot([0], [hv], marker=pl.GROUP_MARKERS["Human"], color=pl.PALETTE["Human"],
                            markersize=9, markeredgecolor="black", markeredgewidth=0.5, zorder=5)
                    human_drawn = True
            ax.set_xticks([0, 1], ["Human", "AI"])
            ax.set_xlim(-0.3, 1.45)
            ax.grid(axis="y", alpha=0.25)
            if ti == 0:
                ax.set_title(cond, fontsize=10)
            if ci == 0:
                ax.set_ylabel(f"{unit}\n↑ more diverse", fontsize=8.5)
                ax.text(-0.42, 1.07, row_lab, transform=ax.transAxes, fontsize=10,
                        fontweight="bold", ha="left")
    handles = [plt.Line2D([], [], marker=pl.GROUP_MARKERS[g], linestyle="", markersize=8,
                          color=pl.PALETTE[g], markeredgecolor="black", markeredgewidth=0.5,
                          markerfacecolor="none" if g == "All AI" else pl.PALETTE[g],
                          label=f"{g} (n=23 of 69)" if g == "All AI" else f"{g} (n=23)")
               for g in ["Human", "Claude", "Gemini", "GPT", "All AI"]]
    fig.legend(handles=handles, fontsize=8.5, ncols=5, loc="upper center", bbox_to_anchor=(0.5, 1.0))
    fig.suptitle(f"{label} · Human → AI slopegraph", y=1.05)
    fig.tight_layout()
    pl.add_direction_badge(fig, "↓ downward slope = narrowing")
    cov_note = ("Coverage's human anchor is the self-benchmark (split-half for proposals, leave-one-out for review "
                "panels), not 1.0. " if metric == "coverage_geometric" else "")
    neg_note = ("Clumping metric plotted as −excess so a downward slope = less diverse, per the Direction Rule. "
                if negate else "")
    _caption(fig, "Left anchor = the human value (shared reference), right anchors = each model and pooled AI, in the "
                  f"metric's natural units; a DOWNWARD slope = narrowing at that stage. {cov_note}{neg_note}"
                  "Top row = generation (proposal sets), bottom row = filtering (paired review-panel means; note the "
                  "compressed absolute range — review panels are homogeneous for everyone, and the drop is in "
                  "consistency, not magnitude). Per-model n=23 vs Human n=23; pooled AI n=23 subsampled from 69. "
                  "Stars: p_raw for pre-registered primaries, p_fdr otherwise, carried from 02/03.")
    pl.save_fig(fig, out / fname)


def fig3_map(df, out):
    metric, params = FACET_SPECS["richness"][0], FACET_SPECS["richness"][1]
    metric_df = _metric_slice(df, metric, params)
    pivot = metric_df.pivot_table(index=["condition", "comparison"], columns="task", values="ratio", aggfunc="mean").reset_index()
    pivot = pivot.dropna(subset=["proposals", "reviews"]) if {"proposals", "reviews"}.issubset(pivot.columns) else pivot.iloc[0:0]
    fig, ax = plt.subplots(figsize=(7, 6.5))
    for comp in CONFIG["comparisons"]:
        g = _model_label(comp)
        pts = {}
        for cond in CONFIG["conditions"]:
            row = pivot[(pivot.condition == cond) & (pivot.comparison == comp)]
            if row.empty:
                continue
            x, y = float(row["proposals"].iloc[0]), float(row["reviews"].iloc[0])
            pts[cond] = (x, y)
            ax.scatter(x, y, s=110, color=pl.PALETTE[g], marker=CONDITION_MARKERS[cond],
                       edgecolors="black", linewidths=0.7, zorder=4)
        chain = [pts[c] for c in CONFIG["conditions"] if c in pts]
        for (x0, y0), (x1, y1) in zip(chain, chain[1:]):
            ax.annotate("", xy=(x1, y1), xytext=(x0, y0),
                        arrowprops=dict(arrowstyle="->", color=pl.PALETTE[g], alpha=0.30, linewidth=1.6))
    ax.axhline(1.0, color=PARITY, linestyle="--", linewidth=1.1)
    ax.axvline(1.0, color=PARITY, linestyle="--", linewidth=1.1)
    xlim, ylim = ax.get_xlim(), ax.get_ylim()
    quad_kw = dict(fontsize=7.5, color="#666666", ha="center", va="center", style="italic")
    ax.text(min(xlim[0] + 0.06, 0.94), min(ylim[0] + 0.04, 0.94), "narrows on BOTH ends\n(the thesis)", **quad_kw)
    ax.text(max(xlim[1] - 0.06, 1.06), min(ylim[0] + 0.04, 0.94), "generates broadly,\nfilters narrowly", **quad_kw)
    ax.text(min(xlim[0] + 0.06, 0.94), max(ylim[1] - 0.04, 1.06), "generates narrowly,\nfilters broadly", **quad_kw)
    ax.text(max(xlim[1] - 0.06, 1.06), max(ylim[1] - 0.04, 1.06), "no narrowing", **quad_kw)
    model_handles = [plt.Line2D([], [], marker="o", linestyle="", markersize=9, color=pl.PALETTE[_model_label(c)],
                                label=_model_label(c)) for c in CONFIG["comparisons"]]
    cond_handles = [plt.Line2D([], [], marker=m, linestyle="", markersize=8, color="#666666", label=c)
                    for c, m in CONDITION_MARKERS.items()]
    ax.legend(handles=model_handles + cond_handles, fontsize=7.5, loc="best", ncols=2)
    ax.set_xlabel("generation ratio (proposals, AI ÷ Human) → more diverse")
    ax.set_ylabel("filtering ratio (reviews, AI ÷ Human) → more diverse")
    ax.set_title("Richness — Vendi VS₁ · 2×2 compression map")
    pl.add_direction_badge(fig, "↗ more diverse on both ends")
    _caption(fig, "Color = model, marker shape = condition, faint arrows = baseline → one_at_a_time → persona. "
                  "Parity lines at 1.0 in ratio space (coverage ratios are already normalized by the split-half/LOO reference).")
    pl.save_fig(fig, out / "fig3_compression_map")


def _stars_from_row(row):
    p = row["p_raw"] if bool(row.get("is_primary", False)) else row["p_fdr"]
    if not np.isfinite(p):
        return ""
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""


def fig4_grid(df, fp, text_version, out):
    """Robustness grid; evenness row re-expressed on the aligned standardized axes
    (redesign spec 4.3) so every row reads up = more diverse."""
    facets = list(FACET_SPECS.keys())
    tasks = ["proposals", "reviews"]
    fig, axes = plt.subplots(len(facets), len(tasks), figsize=(13, 3.1 * len(facets)), sharex="col")
    comps = CONFIG["comparisons"]
    width = 0.2
    for fi, facet in enumerate(facets):
        metric, params = FACET_SPECS[facet]
        for ti, task in enumerate(tasks):
            ax = axes[fi, ti]
            is_evenness = facet == "evenness"
            for ci, comp in enumerate(comps):
                g = _model_label(comp)
                vals, stars = [], []
                for cond in CONFIG["conditions"]:
                    if is_evenness:
                        r = fp[fp["task"].eq(task) & fp["text_version"].eq(text_version)
                               & fp["condition"].eq(cond) & fp["facet"].eq("evenness")
                               & fp["metric"].eq(metric) & fp["group"].eq(g)]
                        vals.append(float(r["z"].iloc[0]) if not r.empty else np.nan)
                        stars.append(str(r["stars"].iloc[0]) if not r.empty else "")
                    else:
                        sub = _whole_only(df)
                        r = sub[sub["task"].eq(task) & sub["facet"].eq(facet) & sub["metric"].eq(metric)
                                & sub["param"].fillna("").eq(params[task])
                                & sub["condition"].eq(cond) & sub["comparison"].eq(comp)]
                        vals.append(float(r["ratio"].iloc[0]) if not r.empty else np.nan)
                        stars.append(_stars_from_row(r.iloc[0]) if not r.empty else "")
                xs = np.arange(len(CONFIG["conditions"])) + (ci - 1.5) * width
                ax.bar(xs, vals, width * 0.92, color=pl.PALETTE[g], edgecolor="black", linewidth=0.4,
                       alpha=0.55 if g == "All AI" else 0.9, hatch="//" if g == "All AI" else None)
                for xv, v, s in zip(xs, vals, stars):
                    if np.isfinite(v) and s:
                        ax.annotate(s, (xv, v), textcoords="offset points", xytext=(0, 2), ha="center", fontsize=7.5)
            ax.axhline(0.0 if is_evenness else 1.0, color=PARITY, linestyle="--", linewidth=1)
            ax.set_xticks(range(len(CONFIG["conditions"])), CONFIG["conditions"], fontsize=8)
            ax.grid(axis="y", alpha=0.25)
            if ti == 0:
                if is_evenness:
                    ylab = "aligned evenness\n(z vs null / aligned δ)\n↑ more even"
                else:
                    ylab = f"{facet}\nAI / Human ratio\n↑ more diverse"
                ax.set_ylabel(ylab, fontsize=8.5)
            if fi == 0:
                ax.set_title("generation (proposals)" if task == "proposals" else "filtering (reviews)")
    handles = [plt.Rectangle((0, 0), 1, 1, facecolor=pl.PALETTE[_model_label(c)],
                             alpha=0.55 if _model_label(c) == "All AI" else 0.9,
                             hatch="//" if _model_label(c) == "All AI" else None, edgecolor="black")
               for c in comps]
    fig.legend(handles, [_model_label(c) for c in comps], fontsize=8.5, ncols=4, loc="upper center",
               bbox_to_anchor=(0.5, 1.015))
    fig.suptitle("Robustness grid · facets × tasks (per-model, never collapsed)", y=1.04)
    fig.tight_layout()
    pl.add_direction_badge(fig, "↑ more diverse (all rows)")
    _caption(fig, "Rows = facets, columns = generation/filtering; EVERY row reads up = more diverse (Direction Rule). "
                  "Ratio rows: AI÷Human, parity at 1. Evenness row (aligned per redesign spec 4.3): generation = "
                  "sign-aligned z vs the pooled-cloud null; filtering = sign-aligned Cliff's δ; parity at 0. Stars: "
                  "p_raw for pre-registered primaries, p_fdr otherwise, carried from 02/03 — no new p-values here. "
                  "Per-model n=23; pooled AI n=23 subsampled from 69.")
    pl.save_fig(fig, out / "fig4_robustness_grid")


def fig5_umaps(out, text_version):
    for condition in CONFIG["conditions"]:
        fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
        specs = [("proposals", "proposal_umap2d.npy", "proposal_master.csv", "Proposal space"),
                 ("reviews", "review_umap2d.npy", "review_master.csv", "Review space")]
        ok = False
        for ax, (task, coords_name, master_name, panel_title) in zip(axes, specs):
            prep = PROJECT_ROOT / "data" / "prepared" / condition / task / text_version
            coords_path, master_path = prep / coords_name, prep / master_name
            if not coords_path.exists() or not master_path.exists():
                ax.text(0.5, 0.5, f"missing prep artifacts:\n{coords_path.name}", ha="center", va="center", fontsize=8)
                ax.set_axis_off()
                continue
            coords = np.load(coords_path)
            master = pd.read_csv(master_path)
            for g in ["Human", "Claude", "Gemini", "GPT"]:
                mask = master["source_group"].eq(g).to_numpy()
                if not mask.any():
                    continue
                ax.scatter(coords[mask, 0], coords[mask, 1], s=22, color=pl.PALETTE[g], alpha=0.75,
                           marker=pl.GROUP_MARKERS.get(g, "o"),
                           edgecolors="white", linewidths=0.3, label=f"{g} (n={int(mask.sum())})")
            ax.set_xlabel("UMAP-1")
            ax.set_ylabel("UMAP-2")
            ax.set_title(panel_title)
            ax.legend(fontsize=7.5)
            ok = True
        fig.suptitle(f"Paired UMAPs · {condition} · {text_version} — illustration only", y=1.0)
        if ok:
            _caption(fig, "Computed for illustration only; ALL metrics are computed in full embedding space (spec 1.1). "
                          "Cached prep-layer coordinates; never refit in a plotting cell.")
        pl.save_fig(fig, out / f"fig5_paired_umaps_{condition}")


def fig6_gradient(df, out):
    metric, params = FACET_SPECS["richness"]
    sub = _metric_slice(df, metric, params)
    sub = sub[sub.comparison == "human_vs_pooled_ai"]
    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    for task, grp in sub.groupby("task"):
        means = grp.groupby("condition")["ratio"].mean().reindex(CONFIG["conditions"])
        ax.plot(CONFIG["conditions"], means, marker="o", linewidth=2,
                color="#1B6CA8" if task == "proposals" else "#B85C00",
                label="generation (proposals)" if task == "proposals" else "filtering (reviews)")
    ax.axhline(1.0, color=PARITY, linestyle="--", linewidth=1)
    ax.annotate("human parity", (0.02, 1.0), textcoords="offset points", xytext=(0, 4), fontsize=8, color=PARITY)
    ax.set_ylabel("pooled AI / Human diversity retained (Vendi VS₁)")
    ax.set_title("Condition gradient · pooled AI (n=23 subsampled from 69)")
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.25)
    pl.add_direction_badge(fig, "↑ more diversity retained")
    _caption(fig, "Rising toward parity left-to-right = persona rescue; the two lines separate whether the rescue is "
                  "stronger for generation or filtering (spec 15.7).")
    pl.save_fig(fig, out / "fig6_condition_gradient")


def _interleaving_stat(inter, condition, stat, *, text_version, comparison="human_vs_pooled_ai", task="proposals"):
    sub = inter[inter["task"].eq(task) & inter["condition"].eq(condition) & inter["stat"].eq(stat)
                & inter["comparison"].eq(comparison) & inter["text_version"].eq(text_version)]
    return float(sub["value"].iloc[0]) if not sub.empty else np.nan


def _tests_row(tests, *, task, facet, metric, param, condition, comparison):
    return tests[tests["task"].eq(task) & tests["facet"].eq(facet) & tests["metric"].eq(metric)
                 & tests["param"].fillna("").eq(param) & tests["condition"].eq(condition)
                 & tests["comparison"].eq(comparison)]


def _geometry_bar(ax, tests, *, facet, metric, param, col, title, ylab, parity, parity_label):
    conds, comps, width = CONFIG["conditions"], CONFIG["comparisons"], 0.22
    for ci, comp in enumerate(comps):
        g = _model_label(comp)
        vals, stars = [], []
        for cond in conds:
            r = _tests_row(tests, task="proposals", facet=facet, metric=metric, param=param,
                           condition=cond, comparison=comp)
            vals.append(float(r[col].iloc[0]) if not r.empty and np.isfinite(r[col].iloc[0]) else np.nan)
            stars.append(_stars_from_row(r.iloc[0]) if not r.empty else "")
        xs = np.arange(len(conds)) + (ci - 1.5) * width
        ax.bar(xs, vals, width * 0.9, color=pl.PALETTE[g], edgecolor="black", linewidth=0.4,
               alpha=0.55 if g == "All AI" else 0.9, hatch="//" if g == "All AI" else None, label=g)
        for xv, v, s in zip(xs, vals, stars):
            if np.isfinite(v) and s:
                ax.annotate(s, (xv, v), textcoords="offset points", xytext=(0, 2), ha="center", fontsize=7)
    ax.axhline(parity, color=PARITY, linestyle="--", linewidth=1)
    ax.annotate(parity_label, (len(conds) - 1.5, parity), textcoords="offset points", xytext=(0, 3),
                fontsize=6.5, color=PARITY, ha="center")
    ax.set_xticks(range(len(conds)), conds, fontsize=7.5)
    ax.set_ylabel(ylab, fontsize=8.5)
    ax.set_title(title, fontsize=9.5)
    ax.grid(axis="y", alpha=0.25)
    ax.legend(fontsize=6.5, ncols=2, loc="best")


def fig_generation_geometry(tests, inter, out, text_version):
    """Fig 3 (draft, canonical): interleaving of human and AI proposals, without a
    projection. For each human proposal, its distance to the nearest other human
    (left dot) vs the nearest AI (right dot, median over the exported size-23 AI
    subsamples), both in the full embedding space. AI as close as the nearest human
    = interleaved; a human whose nearest AI exceeds the human q90 yardstick = fringe.
    Coverage and displacement bar insets carry the quantitative claim."""
    conds = CONFIG["conditions"]
    fig = plt.figure(figsize=(13, 9.0))
    gs = fig.add_gridspec(2, 3, height_ratios=[1.7, 1.0], hspace=0.32, wspace=0.16, top=0.9, bottom=0.07)
    fringe_pct = {}
    for ci, cond in enumerate(conds):
        ax = fig.add_subplot(gs[0, ci])
        prep = PROJECT_ROOT / "data" / "prepared" / cond / "proposals" / text_version
        master = pd.read_csv(prep / "proposal_master.csv")
        grp = master["source_group"].to_numpy()
        hidx = np.where(grp == "Human")[0]
        D = np.load(prep / "proposal_pairwise_cosine_full.npy")
        yq = _interleaving_stat(inter, cond, "human_to_nearest_human_q90", text_version=text_version)
        sub = np.load(prep / "subsample_idx_ai_n23_seed42.npy")
        Dh = D[np.ix_(hidx, hidx)].copy()
        np.fill_diagonal(Dh, np.inf)
        nn_h = Dh.min(axis=1)
        nn_ai_draws = np.empty((sub.shape[0], len(hidx)))
        for d in range(sub.shape[0]):
            nn_ai_draws[d] = D[np.ix_(hidx, sub[d])].min(axis=1)
        nn_ai = np.median(nn_ai_draws, axis=0)
        is_fr = (nn_ai_draws > yq).mean(axis=0) > 0.5
        fringe_pct[cond] = 100.0 * is_fr.mean()
        order = np.argsort(nn_ai)
        yv = np.arange(len(hidx))
        for rank, i in enumerate(order):
            ax.plot([nn_h[i], nn_ai[i]], [rank, rank], color="#bbbbbb", linewidth=0.8, zorder=1)
        ax.scatter(nn_h[order], yv, s=28, color=pl.PALETTE["Human"], marker="o",
                   edgecolors="white", linewidths=0.3, zorder=3, label="nearest other human")
        ax.scatter(nn_ai[order], yv, s=28, color=pl.PALETTE["Claude"], marker="s",
                   edgecolors="white", linewidths=0.3, zorder=3, label="nearest AI")
        fr_ranks = [r for r, i in enumerate(order) if is_fr[i]]
        if fr_ranks:
            ax.scatter(nn_ai[order][is_fr[order]], np.array(fr_ranks), s=120, facecolors="none",
                       edgecolors="#c0504d", linewidths=1.6, zorder=4, label=f"human-only fringe ({int(is_fr.sum())})")
        ax.axvline(yq, color=PARITY, linestyle="--", linewidth=1.1)
        ax.annotate("human q90\nyardstick", (yq, len(hidx) - 1), textcoords="offset points",
                    xytext=(4, -4), fontsize=6.8, color=PARITY, va="top")
        ax.set_title(cond, fontsize=10)
        ax.set_yticks([])
        ax.set_xlabel("cosine distance to nearest neighbor", fontsize=8)
        if ci == 0:
            ax.set_ylabel("human proposals (sorted by nearest AI)", fontsize=8.5)
        ax.legend(fontsize=6.6, loc="lower right", framealpha=0.92)
    axc = fig.add_subplot(gs[1, 0])
    _geometry_bar(axc, tests, facet="coverage", metric="coverage_geometric", param="k=3",
                  col="ai_value", title="Coverage of human idea space", ylab="fraction reached",
                  parity=1.0, parity_label="human self-benchmark")
    axd = fig.add_subplot(gs[1, 1])
    _geometry_bar(axd, tests, facet="displacement", metric="mmd2", param="",
                  col="effect_size", title="Displacement from human region", ylab="MMD²",
                  parity=0.0, parity_label="human split-half floor ≈ 0")
    axn = fig.add_subplot(gs[1, 2]); axn.axis("off")
    note = ("For most human proposals the nearest AI is about as close as the\n"
            "nearest other human: AI sits AMONG the humans, not beside them.\n"
            "The fringe (ringed) are humans no AI approaches within the human\n"
            "yardstick.\n\nhuman-only fringe:  "
            + "   ".join(f"{c}: {fringe_pct[c]:.0f}%" for c in conds))
    axn.text(0.0, 0.95, note, fontsize=8.4, va="top", ha="left", color="#333333")
    fig.suptitle(f"Generation geometry · interleaving of human and AI proposals · {text_version}", y=0.965, fontsize=13)
    _caption(fig, "Left dot = each human proposal's distance to its nearest other human; right dot = its distance to "
                  "the nearest AI (median over the exported size-23 AI subsamples). Computed in the full embedding "
                  "space. Dashed line = the human q90 nearest-neighbor yardstick; a human whose nearest AI falls beyond "
                  "it is fringe. Coverage bars: AI reach at k=3 vs the split-half benchmark. Displacement bars: MMD² "
                  "vs a shuffle null (human split-half floor ≈ 0).")
    pl.save_fig(fig, out / "fig_generation_geometry")


def fig_generation_geometry_umap(tests, inter, out, text_version):
    """Fig 3 (draft): proposal clouds with the human-only fringe highlighted, plus
    coverage and displacement insets. Display-only: UMAP coords and the fringe
    highlight use cached prep artifacts and the exported q90 yardstick; every
    reported statistic still comes from 02 (no metric recomputed here)."""
    conds = CONFIG["conditions"]
    fig = plt.figure(figsize=(13, 8.6))
    gs = fig.add_gridspec(2, 3, height_ratios=[1.55, 1.0], hspace=0.34, wspace=0.14)
    fringe_pct = {}
    for ci, cond in enumerate(conds):
        ax = fig.add_subplot(gs[0, ci])
        prep = PROJECT_ROOT / "data" / "prepared" / cond / "proposals" / text_version
        coords = np.load(prep / "proposal_umap2d.npy")
        master = pd.read_csv(prep / "proposal_master.csv")
        grp = master["source_group"].to_numpy()
        human_idx = np.where(grp == "Human")[0]
        ai_idx = np.where(grp != "Human")[0]
        D = np.load(prep / "proposal_pairwise_cosine_full.npy")
        yq = _interleaving_stat(inter, cond, "human_to_nearest_human_q90", text_version=text_version)
        sub_idx = np.load(prep / "subsample_idx_ai_n23_seed42.npy")
        fringe_prob = np.zeros(len(human_idx))
        for d in range(sub_idx.shape[0]):
            nn = D[np.ix_(human_idx, sub_idx[d])].min(axis=1)
            fringe_prob += (nn > yq)
        fringe_prob /= sub_idx.shape[0]
        is_fringe = fringe_prob > 0.5
        fringe_pct[cond] = 100.0 * is_fringe.mean()
        ax.scatter(coords[ai_idx, 0], coords[ai_idx, 1], s=20, color=pl.PALETTE["Claude"],
                   alpha=0.5, marker="s", edgecolors="white", linewidths=0.3, label="AI (n=69)")
        ax.scatter(coords[human_idx, 0], coords[human_idx, 1], s=34, color=pl.PALETTE["Human"],
                   alpha=0.9, marker="o", edgecolors="white", linewidths=0.4, label="Human (n=23)")
        hf = coords[human_idx][is_fringe]
        ax.scatter(hf[:, 0], hf[:, 1], s=135, facecolors="none", edgecolors=pl.PALETTE["Human"],
                   linewidths=1.7, label=f"human-only fringe ({int(is_fringe.sum())})")
        ax.set_title(cond, fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_xlabel("UMAP-1", fontsize=8)
        if ci == 0:
            ax.set_ylabel("UMAP-2", fontsize=8)
        ax.legend(fontsize=7, loc="best", framealpha=0.9)
    axc = fig.add_subplot(gs[1, 0])
    _geometry_bar(axc, tests, facet="coverage", metric="coverage_geometric", param="k=3",
                  col="ai_value", title="Coverage of human idea space", ylab="fraction reached",
                  parity=1.0, parity_label="human self-benchmark")
    axd = fig.add_subplot(gs[1, 1])
    _geometry_bar(axd, tests, facet="displacement", metric="mmd2", param="",
                  col="effect_size", title="Displacement from human region", ylab="MMD²",
                  parity=0.0, parity_label="human split-half floor ≈ 0")
    axn = fig.add_subplot(gs[1, 2]); axn.axis("off")
    note = ("Low coverage with near-zero displacement means AI fills a\n"
            "smaller region INSIDE the human one, not a shifted region\n"
            "beside it.\n\nhuman-only fringe (illustrative, seed subsamples):\n"
            + "   ".join(f"{c}: {fringe_pct[c]:.0f}%" for c in conds))
    axn.text(0.0, 0.92, note, fontsize=8.6, va="top", ha="left", color="#333333")
    fig.suptitle(f"Generation geometry (UMAP, SI alternate) · proposals · {text_version}", y=0.98, fontsize=13)
    _caption(fig, "UMAP is for display only; coverage, displacement, and the fringe yardstick are computed in the "
                  "full embedding space (02). Fringe = human proposals with no AI proposal within the human q90 "
                  "nearest-neighbor distance, averaged over the exported size-23 AI subsamples. Coverage bars: AI "
                  "reach at k=3 vs the split-half benchmark (1.0). Displacement bars: MMD² vs a shuffle null "
                  "(human split-half floor ≈ 0).")
    pl.save_fig(fig, out / "si_generation_geometry_umap")


def _review_delta(tests, *, text_version, condition, facet, metric):
    r = tests[tests["task"].eq("reviews") & tests["text_version"].eq(text_version)
              & tests["condition"].eq(condition) & tests["facet"].eq(facet)
              & tests["metric"].eq(metric) & tests["comparison"].eq("human_vs_pooled_ai")]
    if r.empty or not np.isfinite(r["effect_size"].iloc[0]):
        return None
    r = r.iloc[0]
    return f"δ={r['effect_size']:+.2f} {_stars_from_row(r)}".strip()


def fig_filtering_panel(tests, out, text_version):
    """Fig 4 (draft): per-proposal review-panel paired slopes for the three facets
    that carry the filtering story. Spread and richness fall (AI panels less
    diverse); coverage rises (each AI panel reaches more of the human review span).
    Rows = facets, columns = conditions; each line is one of the 23 target
    proposals, its human panel value to its matched pooled-AI panel value."""
    facets = [("spread", "mean_pairwise", "mean pairwise distance\n↑ more diverse"),
              ("richness", "vendi", "Vendi VS₁\n↑ more diverse"),
              ("coverage", "coverage_geometric", "coverage of human span\n↑ broader")]
    conds = CONFIG["conditions"]
    fig, axes = plt.subplots(len(facets), len(conds), figsize=(12.5, 9.6), sharey="row")
    for fi, (facet, metric, ylab) in enumerate(facets):
        for ci, cond in enumerate(conds):
            ax = axes[fi, ci]
            paired = pd.read_csv(PROJECT_ROOT / "results" / "tables" / cond / "reviews" / text_version
                                 / "facet_review_paired_long.csv")
            p = paired[paired["field"].eq("whole") & paired["comparison"].eq("human_vs_pooled_ai")
                       & paired["facet"].eq(facet) & paired["metric"].eq(metric)]
            p = p[np.isfinite(p["human_value"]) & np.isfinite(p["ai_value"])]
            for _, r in p.iterrows():
                narrows = (r["ai_value"] < r["human_value"]) if facet != "coverage" else (r["ai_value"] > r["human_value"])
                ax.plot([0, 1], [r["human_value"], r["ai_value"]],
                        color="#c0504d" if (facet != "coverage" and narrows) else pl.PALETTE["Claude"],
                        alpha=0.3, linewidth=1.0, zorder=2)
            if not p.empty:
                ax.plot([0], [p["human_value"].median()], marker="o", color=pl.PALETTE["Human"],
                        markersize=9, markeredgecolor="black", markeredgewidth=0.5, zorder=5)
                ax.plot([1], [p["ai_value"].median()], marker="P", color=pl.PALETTE["All AI"],
                        markersize=10, markeredgecolor="black", markeredgewidth=0.5, zorder=5)
            d = _review_delta(tests, text_version=text_version, condition=cond, facet=facet, metric=metric)
            if d:
                ax.annotate(d, (0.5, 0.95), xycoords="axes fraction", ha="center", va="top",
                            fontsize=8.5, color="#222222")
            ax.set_xticks([0, 1], ["Human\npanel", "AI\npanel"], fontsize=8)
            ax.set_xlim(-0.35, 1.35)
            ax.grid(axis="y", alpha=0.25)
            if fi == 0:
                ax.set_title(cond, fontsize=10)
            if ci == 0:
                ax.set_ylabel(ylab, fontsize=9)
    fig.suptitle(f"Filtering · per-proposal review panels, Human → pooled AI · {text_version}", y=1.0, fontsize=13)
    fig.tight_layout()
    _caption(fig, "Each faint line is one of the 23 target proposals: its human panel value to its matched pooled-AI "
                  "panel value at equal panel size (red = narrows). Large markers are per-condition medians. Spread "
                  "and richness fall for nearly every proposal; coverage rises. δ = paired Cliff's δ (AI − Human) "
                  "from the review tests; stars = p_raw for primaries, p_fdr otherwise. See fig1 filtering block for "
                  "the δ-across-facets summary.")
    pl.save_fig(fig, out / "fig_filtering_panel")



GROUP_TO_COMPARISON = {v: k for k, v in pl.COMPARISON_TO_GROUP.items()}

summary_writes = []
for text_version in CONFIG["text_versions"]:
    out_table = PROJECT_ROOT / "results" / "tables" / "synthesis" / text_version
    out_fig = PROJECT_ROOT / "results" / "figures" / "synthesis" / text_version
    out_table.mkdir(parents=True, exist_ok=True)
    out_fig.mkdir(parents=True, exist_ok=True)
    # Remove superseded old-numbered figure files (regenerable outputs of this notebook).
    for stem in OLD_FIG_STEMS:
        for ext in (".png", ".pdf"):
            stale = out_fig / f"{stem}{ext}"
            if stale.exists():
                stale.unlink()
    sub = ratio_df[ratio_df.text_version == text_version].copy()
    # Aligned standardized columns from the fingerprint CSVs (redesign spec 4.1).
    fp_tv = FP[FP["text_version"].eq(text_version)].copy()
    fp_tv["comparison"] = fp_tv["group"].map(GROUP_TO_COMPARISON)
    gen_z = fp_tv[fp_tv["task"].eq("proposals")][["condition", "facet", "metric", "param", "comparison", "z"]].rename(columns={"z": "z_generation"})
    rev_d = fp_tv[fp_tv["task"].eq("reviews")][["condition", "facet", "metric", "param", "comparison", "z"]].rename(columns={"z": "delta_filtering_aligned"})
    sub = sub.merge(gen_z, on=["condition", "facet", "metric", "param", "comparison"], how="left")
    sub = sub.merge(rev_d, on=["condition", "facet", "metric", "param", "comparison"], how="left")
    tests_sub = T[T["text_version"].eq(text_version)].copy()
    tests_sub["param"] = tests_sub["param"].fillna("")
    sub.to_csv(out_table / "double_compression_summary.csv", index=False)
    if WRITE_FIGURES:
        fig1_fingerprint(FP, out_fig, text_version)
        fig2_slopegraph(sub, out_fig, metric="vendi", params_by_task=FACET_SPECS["richness"][1],
                        label="Richness — Vendi VS₁", fname="fig2_double_compression_slopegraph")
        for facet, (metric, params) in FIG1_SUPPLEMENTS.items():
            fig2_slopegraph(sub, out_fig, metric=metric, params_by_task=params,
                            label=f"{facet.title()} — {metric}" + (" (aligned: Human÷AI)" if metric == "vendi_slope" else ""),
                            fname=f"fig2_supplement_{facet}_{metric}",
                            invert_ratio=(metric == "vendi_slope"))
        for facet, metric, params, label, negate, unit, headline in FIG2B_SPECS:
            fig2b_human_ai(tests_sub, out_fig, facet=facet, metric=metric, params=params,
                           label=label, negate=negate, unit=unit,
                           fname=("fig2b_human_ai_slopegraph" if headline
                                  else f"fig2b_supplement_{facet}_{metric}"))
        fig3_map(sub, out_fig)
        fig4_grid(sub, FP, text_version, out_fig)
        fig5_umaps(out_fig, text_version)
        fig6_gradient(sub, out_fig)
        fig_generation_geometry(T[T.text_version.eq(text_version)],
                                INTER[INTER.text_version.eq(text_version)], out_fig, text_version)
        fig_generation_geometry_umap(T[T.text_version.eq(text_version)],
                                     INTER[INTER.text_version.eq(text_version)], out_fig, text_version)
        fig_filtering_panel(T, out_fig, text_version)
        # SI panel: interleaving / unique-territory check (descriptive; redesign follow-up).
        inter_tv = INTER[INTER["text_version"].eq(text_version)]
        for task in ["proposals", "reviews"]:
            pl.plot_interleaving_si(inter_tv, out_fig / f"si_interleaving_{task}", task=task,
                                    title=f"SI · Interleaving of human and AI {task} · {text_version}")
        # Remove old supplement stems now renamed fig2_supplement_*.
        for old in out_fig.glob("fig1_supplement_*"):
            old.unlink()
    summary_writes.append({"text_version": text_version, "rows": len(sub),
                           "table": str(out_table / "double_compression_summary.csv"), "figures": str(out_fig)})

pd.DataFrame(summary_writes)

,text_version,rows,table,figures
0,rephrased,717,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...
1,original,429,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...
